In [4]:
from pathlib import Path
from selenium import webdriver

TRADINGVIEW_PROFILE_DIR = (
    Path.home()
    / "Documents"
    / "Playground"
    / "selenium_profiles"
    / "tradingview"
)

print(TRADINGVIEW_PROFILE_DIR)

/Users/lol/Documents/Playground/selenium_profiles/tradingview


In [5]:
def open_tradingview_profile_for_login():
    options = webdriver.ChromeOptions()

    # 핵심: 이 경로에 Selenium 전용 Chrome 프로필을 만든다
    options.add_argument(f"--user-data-dir={TRADINGVIEW_PROFILE_DIR}")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    driver.get("https://kr.tradingview.com/")

    print("열린 Chrome 창에서 TradingView에 로그인하세요.")
    print("로그인이 끝나도 Chrome 창을 바로 닫지 말고, 아래 셀에서 확인 후 종료하세요.")

    return driver


login_driver = open_tradingview_profile_for_login()


열린 Chrome 창에서 TradingView에 로그인하세요.
로그인이 끝나도 Chrome 창을 바로 닫지 말고, 아래 셀에서 확인 후 종료하세요.


In [6]:
login_driver.get("https://kr.tradingview.com/symbols/NASDAQ-RXRX/documents/")

input("TradingView에 로그인된 상태로 문서 페이지가 보이면 Enter를 누르세요: ")

login_driver.quit()

print("전용 Chrome 프로필 생성 및 로그인 세션 저장 완료.")


전용 Chrome 프로필 생성 및 로그인 세션 저장 완료.


In [7]:
import time
import json
import re
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
)


URL = "https://kr.tradingview.com/symbols/NASDAQ-RXRX/documents/"
TARGET_YEAR = "2026"

OUTPUT_DIR = Path.cwd()
OUTPUT_TXT = OUTPUT_DIR / "rxrx_2026_transcripts.txt"
OUTPUT_JSON = OUTPUT_DIR / "rxrx_2026_transcripts.json"


def setup_driver(headless=False):
    options = webdriver.ChromeOptions()

    if headless:
        options.add_argument("--headless=new")

    options.add_argument(f"--user-data-dir={TRADINGVIEW_PROFILE_DIR}")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--start-maximized")

    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    return webdriver.Chrome(options=options)


def safe_click(driver, element, sleep_after=0.7):
    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        element,
    )
    time.sleep(0.3)

    try:
        element.click()
    except (ElementClickInterceptedException, StaleElementReferenceException):
        driver.execute_script("arguments[0].click();", element)

    time.sleep(sleep_after)


def auth_wall_exists(driver):
    keywords = [
        "회원가입",
        "가입",
        "로그인",
        "무료로 가입",
        "Sign up",
        "Log in",
        "Join for free",
        "Get started",
    ]

    for dialog in driver.find_elements(By.XPATH, "//div[@role='dialog']"):
        text = dialog.text.strip()
        if any(keyword in text for keyword in keywords):
            return True

    return False


def ensure_logged_in(driver):
    if auth_wall_exists(driver):
        raise RuntimeError(
            "TradingView 로그인/회원가입 모달이 감지되었습니다. "
            "전용 Chrome 프로필 로그인 세션이 풀렸을 가능성이 큽니다."
        )


def close_modal_if_exists(driver):
    close_xpaths = [
        # 사용자가 확인한 TradingView transcript 창 닫기 버튼
        "/html/body/div[8]/div[2]/div/div[1]/div/button",

        # svg/path까지 찍은 XPath의 클릭 가능한 조상 버튼
        "/html/body/div[8]/div[2]/div/div[1]/div/button/span[2]",
        "/html/body/div[8]/div[2]/div/div[1]/div/button/span[2]/*[name()='svg']",
        "/html/body/div[8]/div[2]/div/div[1]/div/button/span[2]/*[name()='svg']/*[name()='path']",

        # 일반 닫기 후보
        "//button[@aria-label='Close']",
        "//button[@aria-label='닫기']",
        "//div[@role='dialog']//button[@aria-label='Close']",
        "//div[@role='dialog']//button[@aria-label='닫기']",
        "//button[contains(@class, 'close')]",
    ]

    for xpath in close_xpaths:
        try:
            element = WebDriverWait(driver, 2).until(
                EC.presence_of_element_located((By.XPATH, xpath))
            )

            # path/svg/span이 잡혀도 실제 클릭은 가장 가까운 button에 수행
            try:
                button = element.find_element(By.XPATH, "./ancestor-or-self::button[1]")
            except Exception:
                button = element

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                button,
            )
            time.sleep(0.2)

            try:
                button.click()
            except Exception:
                driver.execute_script("arguments[0].click();", button)

            time.sleep(1)
            return True

        except Exception:
            continue

    try:
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)
        time.sleep(0.7)
        return True
    except Exception:
        return False



def wait_for_document_articles(driver):
    wait = WebDriverWait(driver, 30)
    wait.until(EC.presence_of_element_located((By.XPATH, "//article")))
    time.sleep(2)

    articles = driver.find_elements(By.XPATH, "//article")

    if not articles:
        raise NoSuchElementException("문서 article을 찾지 못했습니다.")

    return articles


def get_nearest_preceding_year(driver, article):
    """
    article보다 문서상 앞에 있는 가장 가까운 연도 텍스트를 찾는다.
    이전 버전의 el.contains(article) 조건은 상위 컨테이너에서 너무 빨리 멈춰서 제거했다.
    """
    return driver.execute_script(
        """
        const article = arguments[0];
        const targetYears = /^20\\d{2}$/;

        const elements = Array.from(document.body.querySelectorAll('*'));
        let lastYear = null;

        for (const el of elements) {
            if (el === article) {
                break;
            }

            const rect = el.getBoundingClientRect();
            const visible = rect.width > 0 && rect.height > 0;

            if (!visible) {
                continue;
            }

            const text = (el.innerText || el.textContent || '').trim();

            if (targetYears.test(text)) {
                lastYear = text;
            }
        }

        return lastYear;
        """,
        article,
    )


def get_article_title(article, idx):
    text = article.text.strip()

    if not text:
        return f"untitled_{idx}"

    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if not lines:
        return f"untitled_{idx}"

    return lines[0]


def debug_articles(driver, articles):
    print("전체 article 수:", len(articles))
    print("앞쪽 article 샘플:")

    for i, article in enumerate(articles[:5], start=1):
        year = get_nearest_preceding_year(driver, article)
        print(f"\n--- article {i} / preceding year: {year} ---")
        print(article.text[:700])


def get_2026_articles(driver):
    articles = wait_for_document_articles(driver)
    filtered = []

    for article in articles:
        year = get_nearest_preceding_year(driver, article)

        if year == TARGET_YEAR:
            filtered.append(article)

    if not filtered:
        debug_articles(driver, articles)
        raise NoSuchElementException(f"{TARGET_YEAR} article 목록을 찾지 못했습니다.")

    return filtered


def get_2026_articles_info(driver):
    articles = get_2026_articles(driver)
    results = []

    for idx, article in enumerate(articles, start=1):
        results.append(
            {
                "index": idx,
                "title": get_article_title(article, idx),
            }
        )

    return results


def open_article_by_index(driver, article_index):
    articles = get_2026_articles(driver)

    if article_index > len(articles):
        raise IndexError(f"article_index {article_index} out of range")

    article = articles[article_index - 1]
    title = get_article_title(article, article_index)

    buttons = article.find_elements(By.XPATH, ".//button")
    links = article.find_elements(By.XPATH, ".//a")

    target = None

    for btn in buttons:
        label = btn.text.strip().lower()
        if "transcript" in label or "call" in label or "event" in label:
            target = btn
            break

    if target is None and buttons:
        target = buttons[0]

    if target is None and links:
        target = links[0]

    if target is None:
        raise NoSuchElementException(f"클릭할 버튼/링크 없음: {title}")

    safe_click(driver, target, sleep_after=1.5)
    ensure_logged_in(driver)

    return title


def expand_full_text(driver):
    candidate_xpaths = [
        "//div[@role='dialog']//button[contains(., '전체')]",
        "//div[@role='dialog']//button[contains(., '펼치기')]",
        "//div[@role='dialog']//button[contains(., '더 보기')]",
        "//div[@role='dialog']//button[contains(., 'Show more')]",
        "//div[@role='dialog']//button[contains(., 'Read more')]",
    ]

    for xpath in candidate_xpaths:
        try:
            btn = WebDriverWait(driver, 4).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            safe_click(driver, btn, sleep_after=1)
            ensure_logged_in(driver)
            return True
        except Exception:
            continue

    ensure_logged_in(driver)
    return False


def find_transcript_article(driver):
    wait = WebDriverWait(driver, 20)

    transcript_xpaths = [
        "//div[@role='dialog']//article",
        "//div[contains(@data-name, 'news-story-dialog')]//article",
        "(//div[@role='dialog']//*[self::article or self::section])[last()]",
    ]

    for xpath in transcript_xpaths:
        try:
            article = wait.until(
                EC.presence_of_element_located((By.XPATH, xpath))
            )

            if article.text.strip():
                return article

        except TimeoutException:
            continue

    raise NoSuchElementException("상세 transcript 본문을 찾지 못했습니다.")


def parse_paragraph(p_element):
    full_text = p_element.text.strip()

    if not full_text:
        return None

    speaker = ""

    try:
        speaker = p_element.find_element(By.XPATH, ".//strong/span").text.strip()
    except NoSuchElementException:
        pass

    if speaker:
        body = full_text[len(speaker):].strip() if full_text.startswith(speaker) else full_text
        return f"**{speaker}** {body}".strip()

    return full_text


def extract_transcript_lines(driver):
    article = find_transcript_article(driver)
    paragraphs = article.find_elements(By.XPATH, ".//p")

    lines = []

    for p in paragraphs:
        line = parse_paragraph(p)
        if line:
            lines.append(line)

    return lines


def clean_filename(title):
    """
    파일명에 사용할 수 없는 문자 제거.
    macOS/Windows 양쪽을 고려해서 처리.
    """
    invalid_chars = r'[\\/:*?"<>|]'
    filename = re.sub(invalid_chars, "", title)

    filename = filename.replace("\n", " ").replace("\r", " ")
    filename = re.sub(r"\s+", " ", filename).strip()

    if not filename:
        filename = "untitled"

    return filename


def remove_markdown_bold(text):
    return text.replace("**", "")


def save_each_doc_as_txt(all_docs, output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved_files = []

    for doc in all_docs:
        title = doc["title"]
        filename = clean_filename(title) + ".txt"
        filepath = output_dir / filename

        # 같은 제목이 있으면 덮어쓰기 방지
        counter = 2
        while filepath.exists():
            filename = f"{clean_filename(title)}_{counter}.txt"
            filepath = output_dir / filename
            counter += 1

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"# {remove_markdown_bold(title)}\n")
            f.write(f"Year: {doc.get('year', '')}\n")
            f.write(f"Index: {doc.get('index', '')}\n\n")

            for line in doc["lines"]:
                f.write(remove_markdown_bold(line) + "\n\n")

        saved_files.append(filepath)

    return saved_files



def crawl_all_2026_transcripts(headless=False, keep_open_on_error=True):
    driver = setup_driver(headless=headless)
    all_docs = []

    try:
        driver.get(URL)
        time.sleep(5)

        ensure_logged_in(driver)

        article_infos = get_2026_articles_info(driver)
        print(f"{TARGET_YEAR} 문서 수: {len(article_infos)}")

        for info in article_infos:
            idx = info["index"]
            print(f"\n[{idx}] 수집 시작: {info['title']}")

            try:
                title = open_article_by_index(driver, idx)

                expanded = expand_full_text(driver)

                if expanded:
                    print(" -> 전체 텍스트 펼치기 완료")
                else:
                    print(" -> 전체 텍스트 펼치기 버튼 없음 또는 이미 펼쳐짐")

                lines = extract_transcript_lines(driver)

                doc = {
                    "year": int(TARGET_YEAR),
                    "index": idx,
                    "title": title,
                    "lines": lines,
                    "paragraph_count": len(lines),
                }

                all_docs.append(doc)
                print(f" -> 문단 수집 완료: {len(lines)}개")

            except Exception as e:
                print(f" -> 실패: {info['title']} | {e}")

            finally:
                close_modal_if_exists(driver)
                time.sleep(1)

        saved_files = save_each_doc_as_txt(all_docs)
        
        print("\n저장 완료:")
        for filepath in saved_files:
            print(f"- {filepath}")

        driver.quit()
        return all_docs

    except Exception as e:
        print("\n크롤링 중단:")
        print(e)

        if keep_open_on_error:
            print("\n디버깅을 위해 Chrome 창을 닫지 않습니다.")
            print("브라우저를 닫으려면 아래를 실행하세요:")
            print("debug_driver.quit()")
            globals()["debug_driver"] = driver
        else:
            driver.quit()

        raise


In [8]:
docs = crawl_all_2026_transcripts(headless=False, keep_open_on_error=True)


2026 문서 수: 6

[1] 수집 시작: 25th Annual Needham Virtual Healthcare Conference
 -> 전체 텍스트 펼치기 완료
 -> 문단 수집 완료: 75개

[2] 수집 시작: 2026 KeyBanc Capital Markets Healthcare Forum
 -> 전체 텍스트 펼치기 완료
 -> 문단 수집 완료: 47개

[3] 수집 시작: Leerink Global Healthcare Conference 2026
 -> 전체 텍스트 펼치기 완료
 -> 문단 수집 완료: 83개

[4] 수집 시작: TD Cowen 46th Annual Health Care Conference
 -> 전체 텍스트 펼치기 완료
 -> 문단 수집 완료: 73개

[5] 수집 시작: 28th Annual Needham Growth Conference Virtual
 -> 전체 텍스트 펼치기 완료
 -> 문단 수집 완료: 97개

[6] 수집 시작: 44th Annual J.P. Morgan Healthcare Conference
 -> 전체 텍스트 펼치기 완료
 -> 문단 수집 완료: 83개

저장 완료:
- /Users/lol/Documents/GitHub/learning/dev_Python/3. 투자/7. tradingview txt 수집/25th Annual Needham Virtual Healthcare Conference.txt
- /Users/lol/Documents/GitHub/learning/dev_Python/3. 투자/7. tradingview txt 수집/2026 KeyBanc Capital Markets Healthcare Forum.txt
- /Users/lol/Documents/GitHub/learning/dev_Python/3. 투자/7. tradingview txt 수집/Leerink Global Healthcare Conference 2026.txt
- /Users/lol/Documents/GitHub/lea

In [32]:
debug_driver.quit()